# Week 12 — Python Solution Lab
## Resonance & Driven Oscillators

**Companion to `notebooks/Week_12.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_12.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P3` | Phase Lag of a Driven Oscillator | `arctan2` branches; 90° at $\omega_0$ vs the amplitude peak |
| **L2 · Intermediate** | `P8` | Steady-State Amplitude at Three Drive Frequencies | formula validated against a driven simulation |
| **L3 · Challenge** | `P9` | Vibration Isolation for an Optical Table | proves the stated spec infeasible, then designs one |

---

## L1 · Basic — P3: Phase Lag of a Driven Oscillator

> **Problem (Week_12.ipynb, L1 — P3).** A forced oscillator has $\omega_0 = 20$ rad/s, driven at
> $\omega_d = 15$ rad/s with $\gamma = 2.0$ s⁻¹. Calculate the phase lag $\delta$ between the
> driving force and the response.

**Diagram → Principle.** A driven oscillator does not respond in step with the drive. Well below
the natural frequency it nearly keeps up ($\delta \to 0$); at $\omega_d = \omega_0$ it lags by
**exactly** $90^\circ$; far above, it approaches $180^\circ$ out of phase.

> **Two different "resonances" — do not merge them.** The $90^\circ$ crossing happens precisely
> at $\omega_d = \omega_0$, for any damping. The **displacement-amplitude peak** sits somewhere
> else, at $\omega_{\rm peak} = \sqrt{\omega_0^2 - 2\gamma^2}$. Here that is
> $\sqrt{400-8} = 19.799$ rad/s, where the lag is $84.2^\circ$, not $90^\circ$. The two coincide
> only in the zero-damping limit.

**Equation.** $\tan\delta = \dfrac{2\gamma\omega_d}{\omega_0^2 - \omega_d^2}$.

**Hand prediction.** $\tan\delta = 60/175 = 0.3429 \Rightarrow \delta = 18.9^\circ$.

**What Python adds.** The formula has a **sign trap**: above $\omega_0$ the denominator flips sign
and `arctan` silently returns a negative angle, when physically the lag must keep growing toward
$180^\circ$. Using `np.arctan2(2γω_d, ω_0² - ω_d²)` handles the branch correctly — the same
`arctan2` lesson from Week 1, now with physical consequences. The full $\delta(\omega_d)$ curve
makes the $90^\circ$-at-$\omega_0$ crossing obvious.

In [ ]:
# ═══ W12 · L1 · P3 — Phase lag, and why arctan2 matters again ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
w0, wd, gamma = 20.0, 15.0, 2.0

# --- PREDICT ------------------------------------------------------------
num, den = 2*gamma*wd, w0**2 - wd**2
delta = np.degrees(np.arctan2(num, den))
print(f"tan(delta) = 2 gamma w_d / (w0^2 - w_d^2) = {num:.1f}/{den:.1f} = {num/den:.4f}")
print(f"delta = {delta:.2f} deg   (below w0, so a small lag)")

# --- The trap: arctan vs arctan2 above w0 -------------------------------
print("\n  w_d   |  w0^2-w_d^2 |  arctan (naive) | arctan2 (correct)")
for w in (10.0, 15.0, 20.0, 25.0, 40.0):
    n_, d_ = 2*gamma*w, w0**2 - w**2
    naive = np.degrees(np.arctan(n_/d_)) if d_ != 0 else 90.0
    good  = np.degrees(np.arctan2(n_, d_))
    flag  = "  <- WRONG (negative lag!)" if naive < 0 else ""
    print(f"  {w:5.1f} | {d_:11.1f} | {naive:15.2f} | {good:17.2f}{flag}")
print("  Above w0 the denominator goes negative and arctan jumps branch.")
print("  arctan2 keeps delta rising smoothly through 90 deg toward 180 deg.")

# --- VERIFY the three landmark values -----------------------------------
d_low  = np.degrees(np.arctan2(2*gamma*0.01, w0**2 - 0.01**2))
d_res  = np.degrees(np.arctan2(2*gamma*w0,   w0**2 - w0**2))
d_high = np.degrees(np.arctan2(2*gamma*500., w0**2 - 500.**2))
print(f"\n  w_d -> 0    : delta = {d_low:6.2f} deg  (in phase with the drive)")
print(f"  w_d = w0    : delta = {d_res:6.2f} deg  (exactly quadrature -- true for ANY damping)")
print(f"  w_d >> w0   : delta = {d_high:6.2f} deg  (anti-phase; the mass cannot keep up)")
assert abs(d_res - 90.0) < 1e-9
assert d_low < 1.0 and d_high > 179.0

# --- the 90 deg crossing is NOT the amplitude peak ----------------------
w_peak = np.sqrt(w0**2 - 2*gamma**2)
d_peak = np.degrees(np.arctan2(2*gamma*w_peak, w0**2 - w_peak**2))
print(f"\n  CAREFUL -- two distinct 'resonances':")
print(f"    phase crosses 90 deg exactly at w = w0        = {w0:.4f} rad/s")
print(f"    displacement AMPLITUDE peaks at "
      f"sqrt(w0^2 - 2 gamma^2) = {w_peak:.4f} rad/s")
print(f"    and at that peak the lag is {d_peak:.2f} deg, not 90 deg.")
print(f"    They merge only as gamma -> 0. Here gamma = {gamma} splits them by "
      f"{w0 - w_peak:.4f} rad/s.")
assert abs(w_peak - 19.7990) < 1e-3 and abs(d_peak - 84.23) < 0.02

# --- Plot ---------------------------------------------------------------
ws = np.linspace(0.1, 60, 2000)
ds = np.degrees(np.arctan2(2*gamma*ws, w0**2 - ws**2))
amp = 1/np.sqrt((w0**2 - ws**2)**2 + (2*gamma*ws)**2)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.4, 5.6), sharex=True)
ax1.plot(ws, amp/amp.max(), color="#1565c0", lw=2)
ax1.axvline(w0, ls="--", c="crimson"); ax1.set_ylabel("relative amplitude")
ax1.set_title("amplitude peaks just below $\\omega_0$...")
ax2.plot(ws, ds, color="#2e7d32", lw=2)
ax2.axvline(w0, ls="--", c="crimson", label="$\\omega_0$")
ax2.axhline(90, ls=":", c="grey")
ax2.plot(wd, delta, "o", color="#e65100", ms=9, zorder=5, label=f"this problem: {delta:.1f} deg")
ax2.set_xlabel("$\\omega_d$ (rad/s)"); ax2.set_ylabel("phase lag $\\delta$ (deg)")
ax2.set_yticks([0, 45, 90, 135, 180]); ax2.legend(fontsize=9)
ax2.set_title("...and the phase passes through exactly 90 deg there")
for ax in (ax1, ax2): ax.grid(alpha=.3)
plt.suptitle("W12 P3 — driven oscillator response", y=1.01)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(delta - 18.92) < 0.05
print(f"[OK] Matches textbook answer: delta = {delta:.1f} deg")

## L2 · Intermediate — P8: Steady-State Amplitude at Three Drive Frequencies

> **Problem (Week_12.ipynb, L2 — P8).** A $0.50$ kg mass on a spring ($k = 50$ N/m) with
> $b = 1.0$ N·s/m is driven by $F(t) = 3.0\cos(\omega_d t)$ N. Find the steady-state amplitude at
> (a) $\omega_d = 5$, (b) $10$ (resonance), (c) $15$ rad/s.

**A terminology note before we start.** The problem labels $\omega_d = 10$ rad/s "resonance".
Strictly, $10$ rad/s is the **undamped natural frequency** $\omega_0 = \sqrt{k/m}$. With finite
damping the *displacement* amplitude actually peaks slightly lower, at
$\omega_{\rm peak} = \sqrt{\omega_0^2 - 2\gamma^2} = 9.899$ rad/s. The two are routinely
conflated in the light-damping limit, and here they differ by only $1\%$ — but they are not the
same thing, and the sweep at the end of this cell finds the real peak.

**Diagram → Principle.** The steady-state amplitude is set by how close the drive is to $\omega_0$
and how much damping there is.

**Equation.** $A(\omega_d) = \dfrac{F_0/m}{\sqrt{(\omega_0^2-\omega_d^2)^2 + (2\gamma\omega_d)^2}}$.

**Hand prediction.** $\omega_0 = 10$ rad/s, $\gamma = 1.0$ s⁻¹:
$A = 0.0793$, $0.300$, $0.0467$ m.

> ⚠️ **Answer-key discrepancy.** The key prints $0.0800$, $0.300$, $0.0480$ m. The value at
> $\omega_0$ is right, but the two others are the **undamped** approximation — they drop the
> $(2\gamma\omega_d)^2$ term. Keeping it gives $0.0793$ and $0.0467$ m. Away from $\omega_0$ that
> term is a small correction, but it is not zero, and dropping it here is inconsistent with using
> the full formula at $\omega_d = \omega_0$ (where it is the *only* thing keeping $A$ finite).

**What Python adds.** Writing $A(\omega_d)$ once as a function means all three parts are a single
call, and we get the whole resonance curve for free. Then the check that matters: we **integrate
the actual driven ODE** and measure the steady-state amplitude from the late-time signal, which
confirms the full formula — damping term included — really does describe the long-run motion.

In [ ]:
# ═══ W12 · L2 · P8 — Steady-state amplitude: formula vs a real simulation ═══
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
m, k, b, F0 = 0.50, 50.0, 1.0, 3.0
w0    = np.sqrt(k/m)
gamma = b/(2*m)
print(f"omega_0 = {w0:.4f} rad/s,  gamma = {gamma:.4f} 1/s,  Q = {w0/(2*gamma):.3f}")

def amplitude(wd):
    return (F0/m) / np.sqrt((w0**2 - wd**2)**2 + (2*gamma*wd)**2)

# --- PREDICT: all three parts from one function -------------------------
for lbl, wd in (("(a)", 5.0), ("(b)", 10.0), ("(c)", 15.0)):
    note = "  <- at the undamped natural frequency w0" if abs(wd - w0) < 1e-9 else ""
    print(f"{lbl} omega_d = {wd:5.1f} rad/s ->  A = {amplitude(wd):.6f} m{note}")

# --- VERIFY each by integrating the driven ODE and measuring ------------
def steady_amp(wd, n_periods=120):
    T = 2*np.pi/wd
    f = lambda t, y: [y[1], (F0*np.cos(wd*t) - k*y[0] - b*y[1])/m]
    t_end = n_periods*T
    s = solve_ivp(f, [0, t_end], [0.0, 0.0], rtol=1e-9, atol=1e-11, dense_output=True)
    t_tail = np.linspace(t_end - 4*T, t_end, 4000)     # long after transients
    return np.max(np.abs(s.sol(t_tail)[0])), s

print("\n  verification against the integrated equation of motion:")
print(f"  {'w_d':>6s} {'formula (m)':>13s} {'simulated (m)':>15s} {'rel. error':>12s}")
for wd in (5.0, 10.0, 15.0):
    A_f = amplitude(wd)
    A_s, _ = steady_amp(wd)
    rel = abs(A_s - A_f)/A_f
    print(f"  {wd:6.1f} {A_f:13.6f} {A_s:15.6f} {rel:12.2e}")
    assert rel < 5e-3, f"formula and simulation disagree at wd = {wd}"
print("  -> the algebraic amplitude really is the long-run behaviour. [verified]")

# --- Where is the amplitude actually maximum? ---------------------------
w_peak_exact = np.sqrt(w0**2 - 2*gamma**2)
ws = np.linspace(0.1, 25, 6000)
A  = amplitude(ws)
w_peak_num = ws[np.argmax(A)]
print(f"\n  peak of A(w) is at w = {w_peak_num:.4f} rad/s (numerical)")
print(f"  exact sqrt(w0^2 - 2 gamma^2) = {w_peak_exact:.4f} rad/s")
print(f"  NOT exactly w0 = {w0:.4f}. This -- not w0 -- is the true displacement")
print(f"  resonance; damping pulls the peak below the natural frequency.")
assert abs(w_peak_num - w_peak_exact) < 0.02

# --- Plot ---------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(ws, A, color="#1565c0", lw=2)
for wd, col in ((5.0, "#2e7d32"), (10.0, "crimson"), (15.0, "#e65100")):
    ax1.plot(wd, amplitude(wd), "o", color=col, ms=9, zorder=5,
             label=f"$\\omega_d$={wd:.0f}: {amplitude(wd)*100:.2f} cm")
ax1.axvline(w0, ls="--", c="grey", lw=1)
ax1.set_xlabel("$\\omega_d$ (rad/s)"); ax1.set_ylabel("steady-state amplitude (m)")
ax1.set_title("the resonance curve"); ax1.grid(alpha=.3); ax1.legend(fontsize=9)

_, s = steady_amp(10.0, n_periods=25)
tt = np.linspace(0, 25*2*np.pi/10, 4000)
ax2.plot(tt, s.sol(tt)[0], color="crimson", lw=1.4)
ax2.axhline(amplitude(10.0), ls="--", c="k", lw=1.4, label="predicted amplitude")
ax2.axhline(-amplitude(10.0), ls="--", c="k", lw=1.4)
ax2.set_xlabel("t (s)"); ax2.set_ylabel("x (m)")
ax2.set_title("driven at $\\omega_0$ = 10 rad/s: transient -> steady state")
ax2.grid(alpha=.3); ax2.legend(fontsize=9)
plt.suptitle("W12 P8 — driven, damped oscillator", y=1.03)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(amplitude(5.0)  - 0.079298) < 1e-5
assert abs(amplitude(10.0) - 0.300000) < 1e-5
assert abs(amplitude(15.0) - 0.046675) < 1e-5

# --- Reconciling with the printed key -----------------------------------
undamped = lambda wd: (F0/m)/abs(w0**2 - wd**2)
print(f"\nANSWER-KEY NOTE: the key prints 0.0800 / 0.300 / 0.0480 m.")
print(f"  {'w_d':>6s} {'full formula':>14s} {'damping dropped':>17s}  <- the key's values")
for wd in (5.0, 15.0):
    print(f"  {wd:6.1f} {amplitude(wd):14.6f} {undamped(wd):17.6f}")
print(f"  10.0 {amplitude(10.):14.6f} {'infinite':>17s}  (only damping keeps it finite)")
print("  So the key used the undamped approximation away from w0. The simulated")
print("  amplitudes above match the FULL formula, so use 0.0793 / 0.300 / 0.0467 m.")
assert abs(undamped(5.0) - 0.08) < 1e-9 and abs(undamped(15.0) - 0.048) < 1e-9

print(f"\n[OK] CORRECTED: A = {amplitude(5.):.4f}, {amplitude(10.):.4f}, "
      f"{amplitude(15.):.4f} m (verified against the integrated ODE)")

## L3 · Challenge — P9: Vibration Isolation for an Optical Table

> **Problem (Week_12.ipynb, L3 — P9).** A sensitive optical table ($m = 500$ kg) must be isolated
> from floor vibrations in the range $5$–$50$ Hz, mounted on pneumatic isolators modelled as
> springs with adjustable damping. **(a)** What spring constant $k$ gives a natural frequency of
> $2.0$ Hz (below the vibration range)? **(b)** What damping ratio $\zeta = \gamma/\omega_0$
> should be chosen so the transmissibility $T = A_{\rm table}/A_{\rm floor} < 0.05$ at $5$ Hz?
> **(c)** What is $T$ at $50$ Hz with this damping?

**Diagram → Principle.** Base excitation, not force excitation. The ratio of table motion to
floor motion is the **transmissibility**, which is $<1$ only when $\omega_d > \sqrt2\,\omega_0$ —
which is why the mount is deliberately made *soft*.

**Equation.** $k = m\omega_0^2$;
$T = \sqrt{\dfrac{1 + (2\zeta r)^2}{(1-r^2)^2 + (2\zeta r)^2}}$ with $r = f/f_0$.

**Hand prediction (a).** $k = 500(2\pi\cdot2)^2 = 7.896\times10^4$ N/m.

> ## ⚠️ Part (b) as written has no solution
>
> At $5$ Hz with $f_0 = 2$ Hz we have $r = 2.5$, so $(1-r^2)^2 = 27.5625$ and
> $$T(5\,\text{Hz}) = \sqrt{\frac{1+25\zeta^2}{27.5625+25\zeta^2}}.$$
> This is **increasing** in $\zeta$, so its smallest possible value is at $\zeta = 0$:
> $$T_{\min} = \frac{1}{\sqrt{27.5625}} = \frac{1}{5.25} = 0.1905.$$
> That is already **3.8× above the 0.05 target**, and adding damping only makes it worse.
> **No damping ratio satisfies part (b).** The requirement is incompatible with $f_0 = 2$ Hz:
> reaching $T < 0.05$ at $5$ Hz needs $r \ge 4.583$, i.e. $f_0 \le 1.09$ Hz.
>
> So the honest answer to (b) is *"impossible as specified — here is what would be needed
> instead."* The cell below proves the impossibility, then does the design properly under a
> **clearly-labelled substitute criterion**, and answers (c) for that choice.

**What Python adds.** Scanning $\zeta$ shows immediately that (b) has no root, which is easy to
miss when you assume a problem is well-posed. It then solves the substitute criterion
*analytically* rather than picking from a shortlist, and — because this notebook is about
precision — distinguishes two criteria that sound identical but are not:

| Criterion | Meaning | $\zeta$ | $c$ (N·s/m) |
|---|---|---:|---:|
| $T(f_0) \le 5$ | transmissibility **at the nominal natural frequency** | $1/\sqrt{96} = 0.102062$ | 1282.5 |
| $T_{\max} \le 5$ | the **actual peak** of the whole curve | $0.102582$ | 1289.1 |

The peak of $T(r)$ does not sit at $r = 1$: for base excitation it lies slightly *below*, here at
$r = 0.98995$, where $T = 5.024$ rather than $5.000$. The two damping ratios differ by $0.5\%$ —
immaterial for the hardware, but worth being exact about in a notebook that spends its time
insisting on exactness. We compute both and label which is which.

In [ ]:
# ═══ W12 · L3 · P9 — Vibration isolation: a real design trade-off ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
m, f0 = 500.0, 2.0
f_lo, f_hi = 5.0, 50.0             # the band we must isolate

# --- (a) spring constant for a 2 Hz natural frequency -------------------
w0 = 2*np.pi*f0
k  = m*w0**2
print(f"(a) omega_0 = 2 pi f0 = {w0:.4f} rad/s")
print(f"    k = m omega_0^2   = {k:,.1f} N/m = {k/1000:.2f} kN/m")
print(f"    static sag under gravity = m g / k = {m*9.81/k*1000:.2f} mm  (a soft mount)")

# --- The isolation criterion --------------------------------------------
f_cross = np.sqrt(2)*f0
print(f"\n    isolation (T < 1) begins only above sqrt(2)*f0 = {f_cross:.3f} Hz")
print(f"    our band starts at {f_lo:.0f} Hz = {f_lo/f0:.1f} x f0 -> comfortably inside. Good.")
assert f_lo > f_cross

def transmissibility(f, zeta, f0=f0):
    r = np.asarray(f)/f0
    return np.sqrt((1 + (2*zeta*r)**2) / ((1 - r**2)**2 + (2*zeta*r)**2))

# --- (b) FIRST: is the stated requirement achievable at all? ------------
print(f"\n(b) as stated: find zeta such that T < 0.05 at {f_lo:.0f} Hz")
r5 = f_lo/f0
print(f"    r = {f_lo:.0f}/{f0:.0f} = {r5}, so (1 - r^2)^2 = {(1-r5**2)**2:.4f}")
print(f"    T(5 Hz) = sqrt((1 + {(2*r5)**2:.0f} z^2) / ({(1-r5**2)**2:.4f} + "
      f"{(2*r5)**2:.0f} z^2))   -- INCREASING in z")
print(f"\n    {'zeta':>6s} {'T at 5 Hz':>11s}")
for z in (0.0, 0.02, 0.05, 0.10, 0.20, 0.50, 1.0, 2.0):
    print(f"    {z:6.2f} {transmissibility(f_lo, z):11.5f}")
T_floor = transmissibility(f_lo, 0.0)
print(f"\n    The smallest attainable value is at zeta = 0: T = 1/sqrt({(1-r5**2)**2:.4f})"
      f" = {T_floor:.5f}")
print(f"    That is {T_floor/0.05:.2f}x ABOVE the 0.05 target, and damping only raises it.")
print(f"    ==> NO damping ratio satisfies part (b). The spec is infeasible at f0 = 2 Hz.")
assert T_floor > 0.05, "if this ever fails, part (b) would be satisfiable"

r_need = np.sqrt(1 + 1/0.05)          # undamped: 1/(r^2-1) = 0.05
print(f"\n    What WOULD work: undamped, T = 1/(r^2 - 1) < 0.05 needs r > {r_need:.4f},")
print(f"    i.e. f0 < {f_lo}/{r_need:.4f} = {f_lo/r_need:.4f} Hz -- a much softer mount")
print(f"    (k = {m*(2*np.pi*f_lo/r_need)**2:,.0f} N/m instead of {k:,.0f} N/m).")

# --- (b') substitute criterion, SOLVED rather than shortlisted ----------
print(f"\n(b') SUBSTITUTE DESIGN CRITERION (ours, not the problem's):")
print(f"     cap the resonant (steady-state) amplification at 5, then take the SMALLEST")
print(f"     zeta that satisfies it -- T(r) is a steady-state transmissibility, not a")
print(f"     transient start-up figure, so this bounds sustained excitation near f0.")
print(f"     Smallest zeta, because damping costs you at high frequency.")
print(f"     But 'cap it at 5' has TWO readings, and they are not the same number.")

# READING A: transmissibility AT the nominal natural frequency, T(f0) <= 5.
# T(f0) = sqrt((1 + 4 z^2)/(4 z^2)) = 5  ->  1 + 4z^2 = 100 z^2  ->  z = 1/sqrt(96)
z_star = 1/np.sqrt(96)
print(f"\n     (A) T at the nominal natural frequency:  T(f0) <= 5")
print(f"         (1 + 4z^2)/(4z^2) = 25  ->  1 = 96 z^2  ->  zeta = 1/sqrt(96) = {z_star:.6f}")
assert abs(transmissibility(f0, z_star) - 5.0) < 1e-9, "boundary must sit exactly at T = 5"

# READING B: the actual PEAK of the curve, which is not at r = 1.
from scipy.optimize import minimize_scalar, brentq
def T_peak(z):
    r = minimize_scalar(lambda rr: -transmissibility(rr*f0, z),
                        bounds=(0.5, 1.5), method="bounded",
                        options={"xatol": 1e-12}).x
    return transmissibility(r*f0, z), r

Tp_A, r_A = T_peak(z_star)
print(f"\n     (B) the ACTUAL peak of T(r) is not at r = 1 for base excitation:")
print(f"         at zeta = {z_star:.6f} the curve peaks at r = {r_A:.5f} with "
      f"T_max = {Tp_A:.4f}")
print(f"         -- so reading (A) overshoots a true 'peak <= 5' spec by "
      f"{100*(Tp_A/5 - 1):.2f}%.")
z_peak = brentq(lambda z: T_peak(z)[0] - 5.0, 0.09, 0.13, xtol=1e-12)
Tp_B, r_B = T_peak(z_peak)
print(f"         requiring T_max <= 5 exactly gives zeta = {z_peak:.6f} "
      f"(peak {Tp_B:.4f} at r = {r_B:.5f})")
print(f"\n     The two readings differ by {100*(z_peak/z_star - 1):.2f}% in zeta:")
print(f"       (A) zeta = {z_star:.6f}, c = {z_star*2*np.sqrt(k*m):7.1f} N*s/m")
print(f"       (B) zeta = {z_peak:.6f}, c = {z_peak*2*np.sqrt(k*m):7.1f} N*s/m")
print(f"     Immaterial for the hardware; not immaterial for the wording. We quote (A)")
print(f"     below and say so explicitly.")
assert abs(z_star - 0.102062) < 1e-6 and abs(z_peak - 0.102582) < 1e-6
assert Tp_A > 5.0 > transmissibility(f0, z_peak)

fb = np.linspace(f_lo, f_hi, 2000)
print(f"\n     {'zeta':>7s} {'T at f0':>9s} {'worst in band':>15s} {'T at 50 Hz':>12s} {'dB':>8s}")
for z in (0.02, 0.05, z_star, 0.20, 0.50, 1.0):
    tag = "   <- our choice" if abs(z - z_star) < 1e-12 else ""
    print(f"     {z:7.4f} {transmissibility(f0, z):9.2f} "
          f"{transmissibility(fb, z).max():15.5f} {transmissibility(f_hi, z):12.6f} "
          f"{20*np.log10(transmissibility(f_hi, z)):8.1f}{tag}")

print(f"\n     Note zeta = 0.20 also meets the cap, but is strictly worse where it counts:")
print(f"       at 50 Hz  zeta={z_star:.4f} -> {transmissibility(f_hi, z_star):.6f},  "
      f"zeta=0.20 -> {transmissibility(f_hi, 0.20):.6f}  "
      f"({transmissibility(f_hi, 0.20)/transmissibility(f_hi, z_star):.2f}x worse)")
assert transmissibility(f_hi, z_star) < transmissibility(f_hi, 0.20)
assert transmissibility(fb, z_star).max() < 1.0, "must isolate across the whole band"

# --- (c) transmissibility at 50 Hz for the chosen damping ---------------
T50 = transmissibility(f_hi, z_star)
c_rec = z_star*2*np.sqrt(k*m)
print(f"\n(c) at {f_hi:.0f} Hz with zeta = {z_star:.4f}:  T = {T50:.6f} "
      f"({20*np.log10(T50):.1f} dB, i.e. floor motion cut {1/T50:.0f}-fold)")
print(f"    damping coefficient c = 2 zeta sqrt(k m) = {c_rec:,.0f} N*s/m")

# --- Plot ---------------------------------------------------------------
ff = np.logspace(np.log10(0.2), np.log10(200), 2000)
fig, ax = plt.subplots(figsize=(8, 4.6))
for z, col in ((0.02, "#1565c0"), (0.05, "#2e7d32"), (0.20, "#e65100"),
               (0.707, "crimson"), (1.0, "purple")):
    ax.loglog(ff, transmissibility(ff, z), lw=2, color=col, label=f"$\\zeta$ = {z}")
ax.axhline(1, c="k", lw=1, ls="--")
ax.axvline(f0, c="grey", ls=":", lw=1.2)
ax.axvspan(f_lo, f_hi, color="#2e7d32", alpha=.12, label="5-50 Hz band to isolate")
ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("transmissibility  $T = A_{table}/A_{floor}$")
ax.set_title("W12 P9 — every curve crosses T = 1 at $\\sqrt{2}f_0$, no matter the damping")
ax.grid(alpha=.3, which="both"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# --- VERIFY the universal crossover -------------------------------------
for z in (0.02, 0.2, 0.707, 1.0, 3.0):
    assert abs(transmissibility(np.sqrt(2)*f0, z) - 1.0) < 1e-12
print(f"\n  verified: T(sqrt(2) f0) = 1 exactly for EVERY zeta -- damping cannot move")
print("  the crossover, only the shape either side of it.")

# --- CHECK --------------------------------------------------------------
assert abs(k - 78956.8) < 1.0, f"k = {k}"
assert abs(z_star - 0.102062) < 1e-6
assert abs(T50 - 0.008333) < 1e-5
print(f"\n[OK] (a) k = {k/1000:.2f} kN/m for f0 = 2 Hz.")
print(f"     (b) NO zeta satisfies T < 0.05 at 5 Hz -- the floor is {T_floor:.4f} at zeta = 0.")
print(f"         Under the substitute cap T(f0) <= 5: zeta = {z_star:.4f} "
      f"(c = {c_rec:,.0f} N*s/m).")
print(f"     (c) T(50 Hz) = {T50:.6f} ({20*np.log10(T50):.1f} dB) at that damping.")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_12.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
